In [ ]:
import numpy as np
import pandas as pd
import igraph as ig
import vmlab 
from vmlab.models import vmango
from vmlab.processes import harvest

fruit_model = vmango.update_processes({'harvest': harvest.HarvestByQuality})
# fruit_model.visualize() # Descomenta si necesitas verlo

# --- DATOS INICIALES COMPLETOS ---
tree = pd.DataFrame({
    'parent_id': [np.nan, 0, 1, 1],
    'id': [0, 1, 2, 3],
    'topology__is_apical': [1, 1, 0, 1],
    'arch_dev__pot_flowering_date': ['NaT', 'NaT', 'NaT', '2002-09-01'],
    'arch_dev__pot_nb_inflo': [0, 0, 0, 1],
    'arch_dev__pot_nb_fruit': [0, 0, 0, 1],
    'growth__radius_gu': [2, 1, 0.5, 0.5],
    'growth__nb_leaf': [0, 10, 5, 10], 
    'appearance__final_length_gu': [20, 10, 10, 10],
    
    # --- NUEVAS COLUMNAS REQUERIDAS ---
    # Asumimos 1 solo internodo largo por GU para simplificar
    'appearance__final_length_internodes': [ [20.0], [10.0], [10.0], [10.0] ], 
    # Longitudes finales de las hojas (ej. 15cm)
    'appearance__final_length_leaves': [ 
        [],                # GU 0: sin hojas
        [15.0] * 10,       # GU 1: 10 hojas
        [15.0] * 5,        # GU 2: 5 hojas
        [15.0] * 10        # GU 3: 10 hojas
    ], 
    # Longitudes actuales de las hojas (asumimos maduras = final)
     'growth__length_leaves': [ 
        [],                # GU 0: sin hojas
        [15.0] * 10,       # GU 1: 10 hojas
        [15.0] * 5,        # GU 2: 5 hojas
        [15.0] * 10        # GU 3: 10 hojas
    ], 
    # Estado de desarrollo (asumimos maduras)
    'phenology__gu_stage': [4.0, 4.0, 4.0, 4.0] 
    # ------------------------------------
}) 

# --- CONFIGURACIÓN Y EJECUCIÓN (Sin cambios) ---
setup = vmlab.create_setup(
    model=vmango,
    tree=tree,
    start_date='2002-06-01',
    end_date='2003-06-01',
    setup_toml='vmango_fosil.toml', 
    current_cycle=3,
    input_vars={
        'growth__leaf_senescence_enabled': False,
        'geometry__interpretation_freq': 15
    },
    output_vars={ # Reducido para simplificar la prueba
        'topology': { 'adjacency': 'day' },
        'growth': { 'nb_leaf': None },
        'harvest': { 'nb_fruit_harvested': None }
    }
) 

print("Ejecutando simulación con datos iniciales completos...")
ds_out = vmlab.run(setup, fruit_model, geometry=True) # <-- geometry=True
print("Simulación completada.")

# --- VISUALIZACIÓN (Sin cambios) ---
g = ig.Graph.Adjacency([row.tolist() for row in ds_out.topology__adjacency[-1].data.astype(np.int64)])
layout = g.layout_reingold_tilford()
layout.rotate(-180)
ig.plot(g, layout=layout, bbox=(600, 300), **{
    'vertex_size': 1,
    'vertex_label_size': 10,
    'edge_arrow_width': 0.1,
    'vertex_label': [
        f'GU{idx}\nF:{int(ds_out.harvest__nb_fruit_harvested.data[idx])}\nL:{int(ds_out.growth__nb_leaf.data[idx])}' for idx in g.vs.indices
    ]
})

C:\Users\jcarlos.vargas\Documents\GitHub\vmangolab-paleotest\vmlab\vmlab.py:180: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  inputs[attr] = np.array(graph.vs.get_attribute_values(attr)).astype('datetime64[ns]' if 'date' in var_name else np.float32)


ValueError: setting an array element with a sequence.